# Dark Pattern Analyzer -- Stage 1: Fine-tuning

Multilingual multi-label dark pattern classifier for **English, Hindi and Nepali**.

| | |
|---|---|
| Runtime | GPU (a free T4 is sufficient) |
| Expected time | 30-50 minutes end to end |
| Output | `model_v1.zip` -- the artifact bundle for Stage 2 |

### Sections

| # | What | Skippable? |
|---|---|---|
| 1 | Setup and dataset validation | No |
| 2 | **Tokenizer fertility** -- decides the base model | **No** |
| 3 | Baseline (TF-IDF + logistic regression) | No -- it is your floor |
| 4 | Fine-tune the transformer | No |
| 5 | Per-class threshold tuning | No -- free 3-6 F1 points |
| 6 | Evaluation | No |
| 7 | ONNX export (fp32 -- see the note in that section) | No |
| 8 | **Parity test** | **No** |
| 9 | Bundle and download | No |

> **Before you start:** `Runtime > Change runtime type > T4 GPU`

---

### Two decisions this notebook makes on evidence rather than assumption

**Section 2 picks your base model.** Nepali is under-represented in multilingual
pretraining corpora. If mDistilBERT shreds Nepali words into fragments, MuRIL --
pretrained on Nepali explicitly -- becomes the primary model. Ten minutes of
measurement beats a guess you discover was wrong after a full training run.

**Section 8 decides whether the exported model is trustworthy.** int8 quantization
can collapse classes while raising no error at all. That is not a cautionary tale
here -- it happened on this project, to all seven dark classes at once, and the
parity test is the only reason it was caught before Stage 2. Section 7 has the
numbers.

---
## 1. Setup and dataset validation

In [ ]:
# Confirm the GPU is attached. If this reports no GPU, fix the runtime first --
# CPU training takes hours instead of minutes.
import subprocess
try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])
    print(out.decode().strip())
except Exception:
    print('No GPU detected. Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone the repository. Replace with your own URL.
REPO_URL = 'https://github.com/YOUR_USERNAME/dark-pattern-analyzer.git'

import os
if not os.path.exists('/content/dark-pattern-analyzer'):
    !git clone -q $REPO_URL /content/dark-pattern-analyzer

%cd /content/dark-pattern-analyzer/ml
!ls

In [ ]:
# Mount Drive for checkpoints.
#
# Do this. Colab disconnects, and it tends to happen 40 minutes into a run.
# Checkpointing to Drive means a disconnect costs you minutes rather than the
# whole session.
from google.colab import drive
drive.mount('/content/drive')

import pathlib
CKPT = pathlib.Path('/content/drive/MyDrive/dp_checkpoints')
CKPT.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', CKPT)

In [ ]:
# Colab already ships torch. Install only what is missing, and avoid pinning
# aggressively -- fighting Colab's preinstalled versions costs more time than it
# saves.
!pip install -q 'transformers>=4.44' 'datasets>=2.21' 'accelerate>=0.34' \
               'onnx>=1.16' 'onnxruntime>=1.19' 'scikit-learn>=1.5'

import sys
sys.path.insert(0, '/content/dark-pattern-analyzer/ml/src')

import torch, transformers
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
print('cuda        ', torch.cuda.is_available())

In [ ]:
# Validate the dataset BEFORE anything expensive runs.
#
# This asserts template disjointness across train/val/test. It is the most
# important check in the project: the data is template-generated, so if one
# template appears on both sides of a split, the model can score 0.98 by
# memorising skeletons and the number is meaningless.
from ml.dataset import load_all_parts, describe

#DATA = '/content/dark-pattern-analyzer/data/synthetic'
#DATA = '/content/dark-pattern-analyzer/data/synthetic_v2'
DATA = '/content/dark-pattern-analyzer/data/synthetic_v2_1'
parts = load_all_parts(DATA)          # raises on leakage
print(describe(parts))

---
## 2. Tokenizer fertility -- choosing the base model

**Do not skip this.** It runs in a few minutes and it can change what you train.

**Fertility** = subword tokens produced per whitespace word. A tokenizer barely
exposed to Nepali during pretraining shatters Nepali words into fragments, so the
model sees noise instead of morphemes and burns sequence length doing it.

**Decision rule:** if mDistilBERT's Nepali fertility exceeds roughly 1.5x MuRIL's,
switch the primary model to MuRIL and keep mDistilBERT as a documented comparison.

Record the table either way. Nepali tokenizer coverage is a genuine finding about
low-resource NLP and it belongs in your report.

In [ ]:
from ml.tokenizer_fertility import analyse, recommend

fert = analyse(DATA, sample_per_lang=600)
print(recommend(fert))

fert.pivot(index='model_key', columns='lang', values='fertility')

In [ ]:
# Chart it -- this goes straight into your report.
import matplotlib.pyplot as plt

pivot = fert.pivot(index='model_key', columns='lang', values='fertility')
ax = pivot.plot(kind='bar', figsize=(9, 4.5), rot=0)
ax.set_ylabel('subword tokens per word')
ax.set_title('Tokenizer fertility by language (lower is better)')
ax.legend(title='language')
plt.tight_layout(); plt.show()

In [ ]:
# ==> SET YOUR CHOICE HERE, based on the output above.
#
#   'mdistilbert'  distilbert-base-multilingual-cased   (default)
#   'muril'        google/muril-base-cased              (if Nepali fertility is poor)
#   'xlmr'         xlm-roberta-base                     (upper-bound reference)
#   'minilm'       Multilingual-MiniLM-L12-H384         (smallest)
MODEL_KEY = 'mdistilbert'

from ml.config import CANDIDATE_MODELS
print('Training:', CANDIDATE_MODELS[MODEL_KEY])
print()
print('Write your reason down now, while the numbers are in front of you.')
print('It belongs in docs/RESULTS.md and in your report.')

---
## 3. Baseline -- TF-IDF character n-grams + logistic regression

Without a baseline, a macro-F1 of 0.88 is an uninterpretable number.

- Baseline 0.86 -> the transformer added almost nothing, and you should say so.
- Baseline 0.61 -> you have evidence that contextual embeddings genuinely matter.

Character n-grams rather than word n-grams, deliberately: they are script-agnostic,
so one feature extractor covers all three languages without per-language
tokenization. For Devanagari this is a surprisingly strong baseline.

Takes roughly 2-4 minutes on CPU.

In [ ]:
from ml.baseline import run_split
from ml.config import SPLIT_PRIMARY, SPLIT_LEAKY, TrainConfig

cfg = TrainConfig()
cfg.model_name = CANDIDATE_MODELS[MODEL_KEY]

base_primary = run_split(DATA, SPLIT_PRIMARY, cfg)
BASELINE_F1 = base_primary['summary']['macro_f1_dark']
print()
print(f'BASELINE macro_f1_dark = {BASELINE_F1:.4f}')
print('The transformer must beat this on the template-disjoint split.')

In [ ]:
# Quantify the leakage gap using the cheap model.
#
# This measures how much the random split flatters you. Reporting the gap is a
# strength; quietly reporting the random-split number instead is misconduct.
base_leaky = run_split(DATA, SPLIT_LEAKY, cfg)
gap = base_leaky['summary']['macro_f1_dark'] - BASELINE_F1
print()
print(f'LEAKAGE GAP (baseline): {gap:+.4f} macro-F1')
print('Everything reported from here uses the template-disjoint split.')

---
## 4. Fine-tune the transformer

Roughly 10-20 minutes on a T4 for 3 epochs.

**Multi-label, not multi-class.** `problem_type="multi_label_classification"` gives
`BCEWithLogitsLoss` with sigmoid outputs. Softmax would force a single winner, which
is wrong: *"Only 3 left, ends in 10:00"* is genuinely both scarcity and false urgency.

**Trains on `model_input`**, i.e. `[TAG=button] [ROLE=cancel] No thanks`. Structural
context is real signal -- *"No thanks"* in a paragraph is ordinary prose; on a cancel
button beside a bright confirm button it is confirmshaming.

**Early stopping on validation `macro_f1_dark`**, not on loss. Loss keeps falling
while the model memorises template skeletons.

In [ ]:
from ml.train import train

ARTIFACTS = '/content/dark-pattern-analyzer/ml/artifacts/model_v1'

cfg.epochs = 3
cfg.batch_size = 32
cfg.learning_rate = 3e-5

result = train(DATA, ARTIFACTS, cfg, MODEL_KEY)

In [ ]:
# Sanity check against the baseline.
tuned = result['test'].get('test_macro_f1_dark')
print(f'baseline    : {BASELINE_F1:.4f}')
print(f'transformer : {tuned:.4f}  (flat 0.5 threshold, before tuning)')
print(f'improvement : {tuned - BASELINE_F1:+.4f}')
print()

if tuned < BASELINE_F1:
    print('NOT a fair comparison yet -- this is a flat 0.5 cutoff, while the')
    print('baseline fits a decision boundary per class during training.')
    print()
    print('Run section 5 (threshold tuning) BEFORE concluding anything.')
    print('It typically recovers 3-6 macro-F1 points at zero training cost.')
    print()
    print('Only if it still trails after section 6, check in order:')
    print('  1. a threshold pinned to the grid edge (a near-collapsed class)')
    print('  2. label noise -- inspect the false negatives in section 6')
    print('  3. epochs (try 5; early stopping will halt it if it plateaus)')
    print('  4. learning rate -- last resort, 3e-5 is normal for this size')
else:
    print('Good. Thresholds next -- typically another 3-6 points.')

---
## 5. Per-class threshold tuning

The best return-per-effort step in the pipeline: 3-6 macro-F1 points for no training.

A flat 0.5 cutoff assumes every class is equally easy. They are not. `obstruction`
phrasings are diffuse and the model is under-confident on them; `scarcity` phrasings
are formulaic and it is over-confident. One global threshold either floods you with
obstruction misses or scarcity false alarms.

Three profiles are written to `thresholds.json`:

| Profile | Rule | Use for |
|---|---|---|
| `precision` | max F1 subject to precision >= 0.80 | **the extension** |
| `balanced` | max F1 | reported research metrics |
| `recall` | max F1 subject to recall >= 0.80 | annotation assistance |

`precision` is the default because falsely telling a user that an honest site is
manipulating them destroys trust in the tool instantly, whereas a miss is invisible.

Tuned on **validation**. Tuning on test would invalidate your test numbers.

In [ ]:
import json, pathlib
from ml.tune_thresholds import tune

thr = tune(pathlib.Path(ARTIFACTS), DATA)

pathlib.Path(ARTIFACTS, 'thresholds.json').write_text(json.dumps({
    'default_profile': 'precision',
    'tuned_on': 'split_template_disjoint/val',
    'profiles': thr,
}, indent=2))

print()
print('precision-profile thresholds:')
for lab, t in thr['precision']['thresholds'].items():
    print(f'  {lab:<17} {t:.2f}')

---
## 6. Evaluation

Writes `metrics.json` -- the file you commit and quote in your report.

**Always read the per-language table.** An aggregate 0.88 can conceal English at 0.94
and Nepali at 0.71, and the Nepali number is the interesting one.

In [ ]:
from ml.evaluate import evaluate_split

ev = evaluate_split(pathlib.Path(ARTIFACTS), DATA, SPLIT_PRIMARY, 'precision', cfg)

pathlib.Path(ARTIFACTS, 'metrics.json').write_text(
    json.dumps({'primary': ev, 'baseline_macro_f1_dark': BASELINE_F1},
               indent=2, ensure_ascii=False))
print()
print('Wrote metrics.json')

In [ ]:
# Per-class chart for the report.
import pandas as pd

pc = pd.DataFrame(ev['per_class']).T[['precision', 'recall', 'f1', 'support']]
ax = pc[['precision', 'recall', 'f1']].plot(kind='barh', figsize=(9, 5))
ax.set_xlim(0, 1); ax.set_xlabel('score')
ax.set_title('Per-class performance (template-disjoint test, precision profile)')
plt.tight_layout(); plt.show()
pc.round(3)

In [ ]:
# Read the errors. Metrics tell you how well the model does; these tell you what
# it misunderstands, and that is what you actually write about.
print('--- FALSE POSITIVES (flagged benign text) ---')
for e in ev['errors']['false_positives'][:8]:
    print(f"[{e['lang']}] {e['text'][:70]}")
    print(f"    true={e['true']}  pred={e['pred']}")

print()
print('--- FALSE NEGATIVES (missed patterns) ---')
for e in ev['errors']['false_negatives'][:8]:
    print(f"[{e['lang']}] {e['text'][:70]}")
    print(f"    true={e['true']}  pred={e['pred']}")

---
## 7. ONNX export

**Why ONNX rather than shipping PyTorch:** the API needs `onnxruntime` (~50 MB)
instead of `torch` (~2.5 GB), starts in tens of milliseconds instead of seconds,
and runs 2-4x faster on CPU for short sequences.

**Why fp32 and not int8.** Dynamic int8 quantization was attempted and it
destroyed this model. Measured on 200 validation rows:

| Artifact | mean abs prob diff | label agreement | dark classes with 0 positives |
|---|---|---|---|
| fp32 | 0.00000 | 100.00% | 0 of 7 |
| int8, MatMul + embeddings | 0.09181 | 83.81% | **7 of 7** |
| int8, MatMul only | 0.09302 | 84.00% | **7 of 7** |

Excluding the embedding table moved the result by 0.001, so this is not a tuning
problem. Every dark class fell below its threshold and `benign` absorbed 183 of
200 rows, while the smoke test kept printing plausible-looking probabilities.

The bundle is therefore ~950 MB of fp32, which is the honest cost of MuRIL's
197k-token vocabulary -- the same vocabulary that gave it the best Nepali
fertility in section 2. Pruning that vocabulary to three languages would cut the
model to roughly 200 MB and probably let int8 work; that is a Stage 4 task, not a
Stage 1 blocker.

**Two export details that are load-bearing** (both cost several wasted cycles):

1. The `torch.export` (dynamo) exporter is numerically exact. Do **not** switch
   to the legacy exporter with `dynamo=False` -- it freezes a padding branch at
   whatever sequence length the sample batch had, so the graph is silently wrong
   at every other length, and `dynamic_axes` cannot undo it.
2. Request `opset_version=18`. Asking for 17 triggers a downgrade that corrupts
   shape metadata.

`export_fp32` handles both, collapses the sidecar weight file into a single
self-contained `model.onnx`, and raises if the weights escape externally.


In [ ]:
!pip install -q onnx onnxruntime onnxscript

In [ ]:
# Export fp32. int8 is deliberately not used -- see the table above.
import json
import shutil

from ml.export_onnx import export_fp32, verify_loads, write_card

art = pathlib.Path(ARTIFACTS)

# One self-contained file, ~950 MB for MuRIL. Raises if the weights end up in
# an external sidecar, which would break the bundle as soon as it is moved.
fp32 = export_fp32(art, cfg)

final = art / 'model.onnx'
shutil.copy(fp32, final)
fp32.unlink(missing_ok=True)

# Make the bundle describe itself: which dataset trained it, what format it is.
mp = art / 'manifest.json'
blob = json.loads(mp.read_text())
blob['quantized'] = False
blob['quantization'] = 'fp32'
blob['dataset'] = pathlib.Path(DATA).name
mp.write_text(json.dumps(blob, indent=2, ensure_ascii=False))

verify_loads(final, art, cfg)
write_card(art, cfg)

print()
print('fp32 artifact ready. Run section 8 next, before anything else.')
print('Expected: mean |dp| 0.00000 and 100% agreement.')

---
## 8. Parity test -- do not skip

Asserts the ONNX model agrees with the PyTorch model it came from.

This is not a hypothetical risk. On this project, int8 quantization silenced **all
seven** dark classes at once, and **nothing threw an exception**: the export smoke
test still printed sensible-looking probabilities, the API would have started
cleanly and returned well-formed responses, and every element on every page would
have been labelled benign. This test is the only reason it was caught.

Passing means label agreement >= 99%, mean absolute probability difference < 0.02, and
no class whose positive rate collapses.

If it fails, re-export with `--no-quantize` and compare. If fp32 passes, int8 is the
culprit: ship fp32 or exclude the affected layers.

In [ ]:
from ml.parity_test import run as parity_run

ok = parity_run(art, DATA, n=200, profile='precision')
assert ok, 'Parity test failed -- do not proceed to Stage 2 until this passes.'

---
## 9. Bundle and download

Produces `model_v1.zip`, the complete handover to Stage 2.

Unzip it locally into `ml/artifacts/` and point the backend at it with
`DP_MODEL_DIR`. See `docs/SETUP.md`.

In [ ]:
# Verify the bundle is complete before downloading.
required = ['model.onnx', 'label_map.json', 'thresholds.json',
            'manifest.json', 'metrics.json', 'card.md']

missing = [f for f in required if not (art / f).exists()]
if not (art / 'tokenizer').exists():
    missing.append('tokenizer/')

if missing:
    print('MISSING:', missing)
else:
    print('Bundle complete:')
    for f in sorted(art.iterdir()):
        size = f.stat().st_size / 1e6 if f.is_file() else 0
        print(f'  {f.name:<22} {size:>8.2f} MB' if f.is_file() else f'  {f.name}/')

In [ ]:
# Copy to Drive as well -- the download can fail and re-running costs 30 minutes.
import shutil
drive_dest = pathlib.Path('/content/drive/MyDrive/dp_artifacts/model_v1')
drive_dest.parent.mkdir(parents=True, exist_ok=True)
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree(art, drive_dest)
print('Backed up to', drive_dest)

In [ ]:
# Zip and download. Excludes pytorch/ - the backend needs only the ONNX bundle.
zip_path = '/content/model_v1.zip'
!cd {art.parent} && zip -qr {zip_path} model_v1 -x 'model_v1/pytorch/*'
print('zip size:', round(os.path.getsize(zip_path) / 1e6, 1), 'MB')

from google.colab import files
files.download(zip_path)

---
## Stage 1 complete

### Checklist

- [ ] Fertility numbers recorded for all candidate models
- [ ] Base model chosen, with the reason written down
- [ ] Baseline macro-F1 recorded
- [ ] Transformer beats the baseline on the **template-disjoint** split
- [ ] Per-class thresholds tuned on validation
- [ ] Per-language F1 reported; Nepali within ~0.10 of English
- [ ] ONNX exported and quantized
- [ ] **Parity test passed**
- [ ] `model_v1.zip` downloaded and backed up to Drive

### Next

1. Unzip into `ml/artifacts/model_v1/` locally
2. `uv run python -m ml.parity_test --artifacts artifacts/model_v1` to confirm the
   bundle survived the round trip
3. Copy `metrics.json` numbers into `docs/RESULTS.md`
4. Begin Stage 2 -- see `docs/STAGES.md`

### One thing to keep in perspective

These are **synthetic** test numbers. They demonstrate the pipeline works. They are
not evidence the tool works on real websites -- that is Stage 4's hand-annotated gold
set, where a drop to 0.65-0.75 is expected and normal.

That drop is the finding, not a failure. Reporting it honestly, with analysis of which
classes degrade and why, is stronger work than presenting a suspiciously clean 0.99.